In [3]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly.express as px 
import plotly.graph_objects as go 
import plotly.io as pio 
import matplotlib.dates as mdates
from sklearn.preprocessing import LabelEncoder


In [4]:
df_merge = pd.read_csv('data/merged_data.csv')
df_merge.head()

/tmp/ipykernel_26077/1035224579.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_merge = pd.read_csv('data/merged_data.csv')


,adm_0_name,adm_1_name,adm_2_name,full_name,ISO_A0,FAO_GAUL_code,RNE_iso_code,IBGE_code,calendar_start_date,calendar_end_date,...,region,year,month,dayofyear,weekofyear,lag_1,lag_2,lag_3,temperature_c,precipitation_mm
0,BANGLADESH,NaN,NaN,BANGLADESH,BGD,23,BGD,NaN,1980-01-01,1980-12-31,...,SEARO,1980,1,1,1,NaN,NaN,NaN,NaN,NaN
1,BANGLADESH,NaN,NaN,BANGLADESH,BGD,23,BGD,NaN,1985-01-01,1985-12-31,...,SEARO,1985,1,1,1,4.0,NaN,NaN,20.21,0.19
2,BANGLADESH,NaN,NaN,BANGLADESH,BGD,23,BGD,NaN,1986-01-01,1986-12-31,...,SEARO,1986,1,1,1,0.0,4.0,NaN,19.42,0.08
3,BANGLADESH,NaN,NaN,BANGLADESH,BGD,23,BGD,NaN,1987-01-01,1987-12-31,...,SEARO,1987,1,1,1,0.0,0.0,4.0,18.49,0.01
4,BANGLADESH,NaN,NaN,BANGLADESH,BGD,23,BGD,NaN,1988-01-01,1988-12-31,...,SEARO,1988,1,1,53,0.0,0.0,0.0,20.73,0.00


In [5]:
# all column names
df_merge.columns

Index(['adm_0_name', 'adm_1_name', 'adm_2_name', 'full_name', 'ISO_A0',
       'FAO_GAUL_code', 'RNE_iso_code', 'IBGE_code', 'calendar_start_date',
       'calendar_end_date', 'Year', 'dengue_total',
       'case_definition_standardised', 'S_res', 'T_res', 'UUID', 'region',
       'year', 'month', 'dayofyear', 'weekofyear', 'lag_1', 'lag_2', 'lag_3',
       'temperature_c', 'precipitation_mm'],
      dtype='object')

In [6]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41997 entries, 0 to 41996
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   adm_0_name                    41997 non-null  object 
 1   adm_1_name                    39799 non-null  object 
 2   adm_2_name                    2802 non-null   object 
 3   full_name                     41997 non-null  object 
 4   ISO_A0                        41997 non-null  object 
 5   FAO_GAUL_code                 41997 non-null  int64  
 6   RNE_iso_code                  41997 non-null  object 
 7   IBGE_code                     0 non-null      float64
 8   calendar_start_date           41997 non-null  object 
 9   calendar_end_date             41997 non-null  object 
 10  Year                          41997 non-null  int64  
 11  dengue_total                  41997 non-null  int64  
 12  case_definition_standardised  41997 non-null  object 
 13  S

In [7]:

# Convert date columns to datetime
df_merge['calendar_start_date'] = pd.to_datetime(df_merge['calendar_start_date'])
df_merge['calendar_end_date'] = pd.to_datetime(df_merge['calendar_end_date'])

# Sort by country and date (important for lag features)
df_merge = df_merge.sort_values(['ISO_A0', 'calendar_start_date']).reset_index(drop=True)

# ===== FEATURE EXTRACTION FOR TIME SERIES =====

# 1. Time-based features (cyclical encoding for seasonality)
df_merge['month_sin'] = np.sin(2 * np.pi * df_merge['month'] / 12)
df_merge['month_cos'] = np.cos(2 * np.pi * df_merge['month'] / 12)
df_merge['dayofyear_sin'] = np.sin(2 * np.pi * df_merge['dayofyear'] / 365)
df_merge['dayofyear_cos'] = np.cos(2 * np.pi * df_merge['dayofyear'] / 365)

# 2. Season/Quarter encoding
df_merge['quarter'] = df_merge['month'].apply(lambda x: (x-1)//3 + 1)
df_merge['is_rainy_season'] = df_merge['month'].isin([6, 7, 8, 9, 10]).astype(int)  # adjust for your region

# 3. Lag features (if not already present or need more)
for lag in [1, 2, 3, 6, 12]:  # 1, 2, 3 months, 6 months, 1 year
    df_merge[f'dengue_lag_{lag}'] = df_merge.groupby('ISO_A0')['dengue_total'].shift(lag)

# 4. Rolling statistics (moving averages)
for window in [3, 6, 12]:  # 3-month, 6-month, 12-month averages
    df_merge[f'dengue_ma_{window}'] = df_merge.groupby('ISO_A0')['dengue_total'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )
    df_merge[f'dengue_std_{window}'] = df_merge.groupby('ISO_A0')['dengue_total'].transform(
        lambda x: x.rolling(window=window, min_periods=1).std()
    )

# 5. Weather lag features (weather affects dengue with delay)
for lag in [1, 2, 3]:
    df_merge[f'temp_lag_{lag}'] = df_merge.groupby('ISO_A0')['temperature_c'].shift(lag)
    df_merge[f'precip_lag_{lag}'] = df_merge.groupby('ISO_A0')['precipitation_mm'].shift(lag)

# 6. Rolling weather statistics
for window in [3, 6]:
    df_merge[f'temp_ma_{window}'] = df_merge.groupby('ISO_A0')['temperature_c'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )
    df_merge[f'precip_ma_{window}'] = df_merge.groupby('ISO_A0')['precipitation_mm'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

# 7. Year-over-year change (if you have multi-year data)
df_merge['dengue_yoy'] = df_merge.groupby(['ISO_A0', 'month'])['dengue_total'].transform(
    lambda x: x.pct_change(periods=12)  # compare same month last year
)

# 8. Encode country (categorical)
le_country = LabelEncoder()
df_merge['ISO_A0_encoded'] = le_country.fit_transform(df_merge['ISO_A0'])

# 9. Time since start (trend feature)
df_merge['time_index'] = df_merge.groupby('ISO_A0').cumcount() + 1

# 10. Extract year as feature (for trend)
df_merge['year_normalized'] = (df_merge['year'] - df_merge['year'].min()) / (df_merge['year'].max() - df_merge['year'].min())

# ===== SELECT FEATURES FOR MODELING =====

# Drop redundant/unnecessary columns
cols_to_drop = [
    'adm_0_name', 'adm_1_name', 'adm_2_name', 'full_name',  # redundant with ISO_A0
    'FAO_GAUL_code', 'RNE_iso_code', 'IBGE_code',  # ID codes
    'calendar_end_date', 'Year',  # redundant with year/month
    'case_definition_standardised', 'S_res', 'T_res', 'UUID', 'region',  # metadata
    'lag_1', 'lag_2', 'lag_3',  # if you want to use the new lag features instead
]

feature_cols = [
    # Time features
    'year', 'month', 'quarter', 'dayofyear', 'weekofyear',
    'month_sin', 'month_cos', 'dayofyear_sin', 'dayofyear_cos',
    'is_rainy_season', 'year_normalized', 'time_index',
    
    # Location
    'ISO_A0_encoded',
    
    # Lag features
    'dengue_lag_1', 'dengue_lag_2', 'dengue_lag_3', 'dengue_lag_6', 'dengue_lag_12',
    
    # Rolling statistics
    'dengue_ma_3', 'dengue_ma_6', 'dengue_ma_12',
    'dengue_std_3', 'dengue_std_6', 'dengue_std_12',
    
    # Weather features
    'temperature_c', 'precipitation_mm',
    'temp_lag_1', 'temp_lag_2', 'temp_lag_3',
    'precip_lag_1', 'precip_lag_2', 'precip_lag_3',
    'temp_ma_3', 'temp_ma_6',
    'precip_ma_3', 'precip_ma_6',
    
    # Year-over-year
    'dengue_yoy',
]

# Create feature dataframe
df_features = df_merge[feature_cols + ['dengue_total']].copy()

# Drop rows with NaN (from lag features at the beginning)
df_features = df_features.dropna().reset_index(drop=True)

print(f"✅ Feature extraction complete!")
print(f"   Original rows: {len(df_merge)}")
print(f"   Feature rows: {len(df_features)}")
print(f"   Features: {len(feature_cols)}")
print(f"\nFeature columns:")
print(df_features[feature_cols].head())
print(f"\nMissing values:")
print(df_features[feature_cols].isna().sum())

✅ Feature extraction complete!
   Original rows: 41997
   Feature rows: 39989
   Features: 37

Feature columns:
   year  month  quarter  dayofyear  weekofyear  month_sin  month_cos  \
0  1996      1        1          1           1        0.5   0.866025   
1  1999      1        1          1          53        0.5   0.866025   
2  2000      1        1          1          52        0.5   0.866025   
3  2001      1        1          1           1        0.5   0.866025   
4  2002      1        1          1           1        0.5   0.866025   

   dayofyear_sin  dayofyear_cos  is_rainy_season  ...  temp_lag_2  temp_lag_3  \
0       0.017213       0.999852                0  ...       20.45       19.80   
1       0.017213       0.999852                0  ...       19.03       19.49   
2       0.017213       0.999852                0  ...       17.91       19.03   
3       0.017213       0.999852                0  ...       17.97       17.91   
4       0.017213       0.999852                0  